# Day 36 — Practice Questions
### Honest Scores — train/test split, cross-validation, pipelines

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### Four questions to keep asking

- Is this score measured on rows the model **studied**, or rows it **never saw**?
- What is the **baseline**? A score without one means nothing.
- Was this **one split**, or an average of several?
- Did any preparation step look at **all** the rows before the folds were made?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     LeaveOneOut, cross_val_score, cross_validate,
                                     cross_val_predict)
from sklearn.metrics import accuracy_score, confusion_matrix

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

No code needed for most of these. Work them out, then run the cell to check.

### A1 · Fold arithmetic

You have **250 rows** and you run `StratifiedKFold(n_splits=5)`.

1. How many rows are held back in each fold?
2. How many rows does the model study in each fold?
3. How many times is each individual row **held back**?
4. How many times is each individual row **studied**?
5. How many models get fitted in total?

### A2 · How much is one row worth?

You hold back 30% of a dataset and report accuracy on it.

How much does **one row** change the reported accuracy when the dataset has
100 rows? 400 rows? 5,000 rows?

In [2]:
for n in [100, 400, 5000]:
    held = int(n * 0.3)
    print(f"{n:>6} rows  ->  {held:>5} held back  ->  one row is worth {1/held:.4f}")

   100 rows  ->     30 held back  ->  one row is worth 0.0333
   400 rows  ->    120 held back  ->  one row is worth 0.0083
  5000 rows  ->   1500 held back  ->  one row is worth 0.0007


### A3 · Read a cross-validation result

Two models on the same data:

| | fold scores |
|---|---|
| **Model P** | 0.81, 0.79, 0.82, 0.80, 0.78 |
| **Model Q** | 0.95, 0.62, 0.88, 0.71, 0.84 |

Work out each one's mean and standard deviation. Which would you deploy, and why?

In [3]:
P = np.array([0.81, 0.79, 0.82, 0.80, 0.78])
Q = np.array([0.95, 0.62, 0.88, 0.71, 0.84])
for name, a in [("P", P), ("Q", Q)]:
    print(f"model {name}:  mean {a.mean():.4f}   sd {a.std(ddof=1):.4f}   "
          f"range {a.min():.2f} to {a.max():.2f}")

model P:  mean 0.8000   sd 0.0158   range 0.78 to 0.82
model Q:  mean 0.8000   sd 0.1332   range 0.62 to 0.95


### A4 · Which of these leak?

For each step, say whether doing it **before** splitting leaks, and why.

1. `StandardScaler` on all the rows
2. Dropping a column that is 95% empty
3. `SelectKBest(f_classif, k=10)` on all the rows
4. Filling missing ages with the median age of all the rows
5. Trying `max_depth` from 1 to 20 and keeping whichever scored best
6. `OneHotEncoder` on all the rows

### A5 · Why is leave-one-out's ± meaningless?

A model run with `LeaveOneOut()` on 200 rows reports **mean 0.8500, sd 0.3580**.

Explain where the 0.3579 comes from, and what it would be if the model had got
170 of the 200 right in a completely different way.

In [4]:
n = 200
for correct in [170, 170]:
    folds = np.array([1.0]*correct + [0.0]*(n-correct))
    print(f"{correct} right of {n}:  mean {folds.mean():.4f}   sd {folds.std(ddof=1):.4f}")

p = 0.85
print(f"\nformula for ANY list of {n} zeros/ones with mean {p}: "
      f"{np.sqrt(p*(1-p)*n/(n-1)):.4f}")

170 right of 200:  mean 0.8500   sd 0.3580
170 right of 200:  mean 0.8500   sd 0.3580

formula for ANY list of 200 zeros/ones with mean 0.85: 0.3580


### A6 · The sorted file

A hospital exports 500 patient records, sorted by outcome: all the survivors first, then
all the deaths. 20% died.

You run `KFold(n_splits=5)` with no other arguments.

1. How many rows in each fold?
2. What is the death rate inside fold 1? Inside fold 5?
3. What will a model that predicts "survived" for everybody score on each fold?

### A7 · Does this need a final test set?

For each, say whether cross-validation alone is honest, or whether you also need a test
set that nothing has touched.

1. Fit one logistic regression. Report its 5-fold CV score.
2. Fit logistic regression, a decision tree and kNN. Report the best CV score of the three.
3. Try `C` = 0.01, 0.1, 1, 10, 100. Report the best CV score.
4. Fit one model, report its CV score, and *then* decide it is good enough to deploy.

### A8 · Which splitter?

Pick the right cross-validator, and say in one line why.

1. 900 loan applications, 38% approved, rows in no particular order
2. House prices over time; you want to predict next quarter
3. 4,000 X-rays from 800 patients (several images each), predicting a fracture
4. 200 emails, 4% spam

---
# Part B — Find the mistake

Four short programs. Each runs cleanly and reports a number that is wrong.

In [5]:
# ---------------------------------------------------------------
# 400 clinic appointments. B and C both use `clinic`.
# ---------------------------------------------------------------
r = np.random.default_rng(3650)
N = 400

age               = r.normal(34, 14, N).clip(5, 85).round(0)
lead_time_days    = r.uniform(0, 60, N).round(0)
previous_no_shows = r.poisson(0.7, N).clip(0, 6).astype(float)
distance_km       = r.normal(9, 6, N).clip(0.5, 45).round(1)
reminder          = r.choice(["none", "sms", "call"], N, p=[.35, .45, .20])

rb = np.where(reminder == "none", 0.45, np.where(reminder == "sms", 0.0, -0.40))
z = (-4.1 + 3.5*(0.022*lead_time_days + 0.55*previous_no_shows
                 - 0.012*age + 0.020*distance_km + rb) + r.normal(0, 0.45, N))
no_show = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

clinic = pd.DataFrame({"age": age, "lead_time_days": lead_time_days,
                       "previous_no_shows": previous_no_shows,
                       "distance_km": distance_km, "reminder": reminder,
                       "no_show": no_show})
clinic.loc[r.uniform(0, 1, N) < 0.10, "distance_km"] = np.nan   # some addresses missing

NUM = ["age", "lead_time_days", "previous_no_shows", "distance_km"]
CAT = ["reminder"]
print("rows:", len(clinic), "  no-show rate:", round(clinic.no_show.mean(), 4))
print("missing distances:", int(clinic.distance_km.isna().sum()))
clinic.head()

rows: 400   no-show rate: 0.395
missing distances: 28


,age,lead_time_days,previous_no_shows,distance_km,reminder,no_show
0,55.0,54.0,1.0,NaN,call,0
1,34.0,7.0,1.0,0.5,sms,0
2,46.0,25.0,1.0,0.5,sms,0
3,44.0,5.0,1.0,5.4,none,1
4,33.0,9.0,1.0,9.8,sms,0


In [6]:
# a filled, numeric copy for the simpler questions
flat = pd.get_dummies(clinic[NUM + CAT], columns=CAT, drop_first=True).astype(float)
flat["distance_km"] = flat["distance_km"].fillna(flat["distance_km"].median())
target = clinic.no_show
print(list(flat.columns))

['age', 'lead_time_days', 'previous_no_shows', 'distance_km', 'reminder_none', 'reminder_sms']


### B1 · "Our model is 100% accurate"

In [7]:
X_train, X_hidden, y_train, y_hidden = train_test_split(
    flat, target, test_size=0.3, random_state=0, stratify=target)

tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

print(f"accuracy: {tree.score(X_train, y_train):.4f}")
print()
print("REPORT: 'The no-show model is 100% accurate. Ready to deploy.'")

accuracy: 1.0000

REPORT: 'The no-show model is 100% accurate. Ready to deploy.'


### B2 · The seed that was tried until it worked

In [8]:
X_train, X_hidden, y_train, y_hidden = train_test_split(
    flat, target, test_size=0.3, random_state=12, stratify=target)

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
print(f"accuracy on held-back rows: {model.score(X_hidden, y_hidden):.4f}")
print()
print("REPORT: 'The model reaches 86.7% on unseen appointments.'")

accuracy on held-back rows: 0.8667

REPORT: 'The model reaches 86.7% on unseen appointments.'


### B3 · The folds that were never shuffled

In [9]:
# the clinic's export tool writes the no-shows at the end of the file
exported = clinic.sort_values("no_show").reset_index(drop=True)
Xe = pd.get_dummies(exported[NUM + CAT], columns=CAT, drop_first=True).astype(float)
Xe["distance_km"] = Xe["distance_km"].fillna(Xe["distance_km"].median())
ye = exported.no_show

scores = cross_val_score(LogisticRegression(max_iter=2000), Xe, ye, cv=KFold(5))
print("fold scores:", np.round(scores, 4))
print(f"reported   : {scores.mean():.4f}")

fold scores: [0.775  0.8625 0.7625 0.5375 0.5375]
reported   : 0.6950


### B4 · The preparation that happened too early

In [10]:
analyst = clinic.copy()

#  the analyst's "cleaning" step, run on the whole file before any splitting
analyst["distance_km"] = analyst["distance_km"].fillna(analyst["distance_km"].median())
Xa = pd.get_dummies(analyst[NUM + CAT], columns=CAT, drop_first=True).astype(float)
Xa[NUM] = StandardScaler().fit_transform(Xa[NUM])

sc = cross_val_score(LogisticRegression(max_iter=2000), Xa, analyst.no_show,
                     cv=StratifiedKFold(5, shuffle=True, random_state=0))
print(f"cross-validated accuracy: {sc.mean():.4f}")

cross-validated accuracy: 0.8050


---
# Part C — Write the code

Five tasks on the clinic data.

### C1 · Baseline first

Before any model: what fraction of appointments were no-shows, and what is the accuracy
of the laziest possible model? Also break the no-show rate down by `reminder`.

In [11]:
# your code here

### C2 · Cross-validate properly

Build a `ColumnTransformer` + `LogisticRegression` pipeline that fills the missing
distances with a median, scales the numbers, and one-hot encodes `reminder`.

Cross-validate it with 5 stratified shuffled folds. Report the five fold scores, the mean,
the standard deviation, **and** the training score, so you can see the gap.

In [12]:
# your code here

### C3 · How many folds?

Run the same pipeline with 3, 5 and 10 folds. Report each mean and standard deviation, and
say which you would use here and why.

In [13]:
# your code here

### C4 · Fit it on everything and read the coefficients

Fit the pipeline on all 400 rows. Print each column the pipeline built, its coefficient,
and what it does to the odds. Which column matters most, and what should the clinic
actually do?

In [14]:
# your code here

### C5 · Save it and use it on three new patients

Fit the pipeline, save it with `joblib`, load it back, and score these three:

| | age | lead time | previous no-shows | distance | reminder |
|---|---|---|---|---|---|
| **A** | 23 | 52 days | 3 | 22 km | none |
| **B** | 61 | 4 days | 0 | 3 km | call |
| **C** | 35 | 21 days | 1 | **unknown** | sms |

Patient C's address is missing — make sure that does not break anything.

In [15]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Write k-fold cross-validation yourself

Write `my_cross_val_score(model, X, y, k=5, seed=0)` using **no** scikit-learn
cross-validation tools. You may use `clone` from `sklearn.base`.

It should:

- shuffle the row numbers, then cut them into `k` groups as evenly as possible
  (with 400 rows and k=3, the groups must be 134/133/133 — not 133/133/134 with one lost)
- for each group: fit a **fresh** copy of the model on the other rows, score it on that group
- return the `k` scores as an array

Then prove it matches `cross_val_score` with `KFold(k, shuffle=True, random_state=seed)`.

**The hard part is matching scikit-learn's grouping exactly.** Read `KFold`'s rule: with
`n` rows and `k` folds, the first `n % k` folds get `n//k + 1` rows and the rest get
`n//k`.

In [16]:
demo_X = flat.values
demo_y = target.values
print(demo_X.shape, demo_y.shape)

(400, 6) (400,)


In [17]:
# your function here

### D2 · Audit a model the company is already using

A company has deployed a model that predicts which employees will leave within a year.
HR uses its output to decide who gets a retention conversation. The analyst's report says:

> *"Cross-validated accuracy **0.906** on 2,000 employee records, using 44 features. The
> model is in production and performing well."*

The code below is the analyst's, reproduced exactly. It runs without error and the 0.906
is real.

**Your tasks:**

1. Run it and confirm the 0.906.
2. Find **three separate things wrong** with it.
3. Prove each with numbers.
4. Write a recommendation for the Head of HR — not a lecture on methodology, a decision.

In [18]:
r = np.random.default_rng(3660)
N = 2000

department   = r.choice(["sales", "eng", "support", "ops"], N, p=[.28, .34, .22, .16])
tenure_years = r.uniform(0.2, 15, N).round(1)
salary_band  = r.integers(1, 8, N).astype(float)
commute_min  = r.normal(18, 11, N).clip(1, 80).round(0)
last_rating  = r.normal(3.4, 0.8, N).clip(1, 5).round(1)

db = np.where(department == "sales", 0.6,
     np.where(department == "support", 0.35,
     np.where(department == "eng", -0.2, 0.0)))
z = (0.70 - 0.16*tenure_years - 0.22*salary_band + 0.020*commute_min
     - 0.55*last_rating + db + r.normal(0, 0.4, N))
left = (r.uniform(0, 1, N) < sigmoid(z)).astype(int)

hr = pd.DataFrame({"tenure_years": tenure_years, "salary_band": salary_band,
                   "commute_minutes": commute_min, "last_rating": last_rating,
                   "department": department})
#  40 engagement-survey questions the analyst also had available
survey = r.normal(0, 1, (N, 40))
for i in range(40):
    hr[f"survey_q{i+1}"] = survey[:, i].round(3)
hr["left_company"] = left

#  the HR system exports records grouped by department
hr = hr.sort_values("department").reset_index(drop=True)
hr.head()

,tenure_years,salary_band,commute_minutes,last_rating,department,survey_q1,survey_q2,survey_q3,survey_q4,survey_q5,...,survey_q32,survey_q33,survey_q34,survey_q35,survey_q36,survey_q37,survey_q38,survey_q39,survey_q40,left_company
0,5.9,4.0,28.0,3.5,eng,0.707,0.143,0.880,-0.197,-2.562,...,0.925,1.401,0.629,0.329,-0.898,-0.971,0.238,-0.419,2.013,0
1,2.6,7.0,6.0,2.8,eng,0.274,0.992,0.081,-1.872,1.530,...,-0.593,1.211,-1.323,-0.977,0.126,0.101,-0.912,-1.280,-0.879,0
2,4.3,2.0,7.0,3.1,eng,0.372,2.012,0.841,1.384,-0.030,...,-0.158,-0.536,-0.021,0.935,0.771,-0.464,3.125,1.143,-1.263,0
3,3.9,4.0,36.0,4.6,eng,0.775,1.222,0.929,0.812,-0.016,...,0.298,2.100,0.437,0.491,-0.057,1.384,-0.820,-0.632,0.626,0
4,8.5,4.0,1.0,5.0,eng,-0.194,-0.818,1.184,1.535,0.279,...,0.439,0.473,0.433,-1.533,-0.835,-0.104,0.313,1.477,-0.945,0


In [19]:
#  ---- the analyst's code, exactly as written ----
features = [c for c in hr.columns if c not in ("left_company", "department")]
Xh = hr[features].values
yh = hr.left_company.values

Xh_scaled = StandardScaler().fit_transform(Xh)
best10 = SelectKBest(f_classif, k=10).fit(Xh_scaled, yh)

report = cross_val_score(LogisticRegression(max_iter=2000),
                         best10.transform(Xh_scaled), yh, cv=KFold(5))

print(f"cross-validated accuracy: {report.mean():.4f}")
print(f"fold scores             : {np.round(report, 4)}")

cross-validated accuracy: 0.9060
fold scores             : [0.94   0.9525 0.88   0.8425 0.915 ]


In [20]:
# your investigation here

---
## Before you hand it in

- Is every score you quote measured on rows the model **never saw**?
- Is every accuracy sitting **next to its baseline**?
- Did you report a **mean and a spread**, not a single number?
- Is `shuffle=True` on, and `stratify` where it belongs?
- Is every preparation step **inside** the Pipeline?
- If you compared several models, did you keep a **final test set** back?

Five numbers worth carrying away:

- A decision tree scored **1.0000** on the appointments it studied and **0.7250** on the
  ones it had not — the worst of the three models on the only column that counts.
- Twenty splits of the same 400 appointments ranged over **11.7 points**. The report
  quoted the highest.
- A file sorted by outcome dropped the reported accuracy from **0.8125 to 0.6825**, with
  three folds containing no no-shows at all.
- Scaling and median-filling before the split leaked **+0.0000**. Adding 300 junk columns
  and letting `SelectKBest` choose leaked **+0.0225**.
- A deployed HR model with **0.906 accuracy** matched its baseline exactly and caught
  **2 of 187** leavers.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*